In [0]:
# Bronze — Source 02: Debezium CDC
# CDC envelope: {before, after, op, source, ts_ms}
# op: c=create, u=update, d=delete, r=read(snapshot)
import sys
sys.path.append('/Workspace/Users/sutharripal26@gmail.com/ecommerce-lakehouse/pipelines/bronze/shared')
from bronze_utils import get_watermark, update_watermark

RAW_BUCKET = 's3://ecommerce-lakehouse-467091806172-raw-01'
SOURCE = '02_debezium_cdc'
TARGET_TABLE = 'bronze.src_02_cdc.events'
MERGE_KEY = 'cdc_event_id'
PATH = f'{RAW_BUCKET}/source=02_debezium_cdc/debezium.ecommerce.*/year=*/month=*/day=*/'


In [0]:
from pyspark.sql.functions import col, lit, max as spark_max, regexp_extract, md5, concat_ws
from pyspark.sql.types import *
import re as re_lib

watermark = get_watermark(spark, SOURCE)
print(f'[{SOURCE}] Watermark: {watermark}')

raw_df = spark.read.text(PATH) \
    .filter(col('_metadata.file_modification_time') > lit(watermark))

if raw_df.count() == 0:
    print(f'[{SOURCE}] No new files — skipping')
    dbutils.notebook.exit('No new data')

from pyspark.sql.functions import udf

# Debezium Struct string format parser
# Format: "Struct{after=Struct{field1=val1,...},op=c,source=Struct{...}}"
@udf(returnType=StringType())
def extract_op(s):
    if not s: return None
    try:
        # unwrap outer quotes
        s = s.strip('"')
        m = re_lib.search(r',op=([a-z])', s)
        return m.group(1) if m else None
    except: return None

@udf(returnType=StringType())
def extract_table(s):
    if not s: return None
    try:
        s = s.strip('"')
        m = re_lib.search(r'name=debezium\.ecommerce,ts_ms=\d+,snapshot=\w+,db=\w+,sequence=\[.*?\],ts_us=\d+,ts_ns=\d+,schema=public,table=(\w+)', s)
        return m.group(1) if m else None
    except: return None

@udf(returnType=LongType())
def extract_ts_ms(s):
    if not s: return None
    try:
        s = s.strip('"')
        m = re_lib.search(r'ts_ms=(\d+)', s)
        return int(m.group(1)) if m else None
    except: return None

@udf(returnType=StringType())
def extract_after(s):
    if not s: return None
    try:
        s = s.strip('"')
        m = re_lib.search(r'after=Struct\{([^}]+)\}', s)
        return m.group(1) if m else None
    except: return None

@udf(returnType=StringType())
def make_event_id(ts, op, after):
    if not ts or not op: return None
    import hashlib
    raw = f'{ts}|{op}|{after or ""}'
    return hashlib.md5(raw.encode()).hexdigest()

df = raw_df \
    .withColumn('cdc_op', extract_op(col('value'))) \
    .withColumn('cdc_table', extract_table(col('value'))) \
    .withColumn('cdc_ts_ms', extract_ts_ms(col('value'))) \
    .withColumn('after_raw', extract_after(col('value'))) \
    .withColumn('cdc_event_id', make_event_id(col('cdc_ts_ms').cast('string'), col('cdc_op'), col('after_raw'))) \
    .withColumn('raw_value', col('value')) \
    .filter(col('cdc_op').isNotNull())

row_count = df.count()
print(f'[{SOURCE}] {row_count} CDC events parsed')

ops = df.groupBy('cdc_op').count().collect()
for r in ops:
    print(f'  op={r["cdc_op"]}: {r["count"]} events')

spark.sql('CREATE SCHEMA IF NOT EXISTS bronze.src_02_cdc')

if spark.catalog.tableExists(TARGET_TABLE):
    from delta.tables import DeltaTable
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    dt.alias('t').merge(df.alias('s'), f't.{MERGE_KEY} = s.{MERGE_KEY}') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print('MERGE complete')
else:
    df.write.format('delta').mode('overwrite') \
        .option('mergeSchema', 'true').saveAsTable(TARGET_TABLE)
    print('Initial load complete')

latest_ts = raw_df.select(spark_max('_metadata.file_modification_time')).collect()[0][0]
update_watermark(spark, SOURCE, latest_ts, row_count)
print(f'Watermark updated to {latest_ts}')


In [0]:
count = spark.sql(f'SELECT COUNT(*) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
print(f'{TARGET_TABLE}: {count} rows')
spark.sql(f"SELECT * FROM bronze.pipeline.watermarks WHERE source = '{SOURCE}'").show()
print('\nSample CDC events:')
spark.sql(f'SELECT cdc_op, cdc_table, cdc_ts_ms FROM {TARGET_TABLE} LIMIT 5').show()
